# Tutorial - WebSensors Flow - Text Classification with GenAI

This notebook presents a complete GenAI-based text classification pipeline using **WebSensors Flow**.

The tutorial uses a small subset of the **DMOZ Health** dataset, which contains textual descriptions of web pages related to health topics. Each record has a text and an associated category. The goal is to classify health-related texts using a Large Language Model through an **in-context learning** strategy.

In this example, the model does not learn from the dataset through traditional parameter optimization. Instead, classification is performed by prompting an LLM with task instructions, candidate categories, and examples when available.

This approach is useful for scenarios where labeled data is limited, where fast experimentation is needed, or where the goal is to compare GenAI-based classification with traditional supervised learning methods.

The pipeline is organized as a single end-to-end flow with four main steps:

1. data ingestion from the DMOZ Health dataset;
2. text preprocessing and prompt configuration;
3. GenAI-based classification using an LLM classifier;
4. evaluation, telemetry aggregation, and result serialization.

The GenAI model is wrapped as a scikit-learn-compatible classifier with `fit` and `predict` methods. This makes the LLM-based classifier behave like a regular model inside the pipeline, while keeping GenAI-specific details available for observability.


This design keeps the pipeline lightweight, modular, and fully customizable. Each project can decide what should be monitored, such as dataset information, prompt configuration, model name, number of classified texts, latency, token usage, prediction distribution, evaluation metrics, errors, and generated artifacts.

The notebook defines the step classes directly in the cells and executes them one by one before running the complete flow. This makes the execution easier to inspect, debug, and understand before moving to a more automated or production-oriented scenario.

## 1. Imports and workspace

This first cell prepares the notebook environment.

The important point is that the notebook imports `websensors-flow` objects such as `PipelineStep`, `StepResult`, and `MetricRecord`. These are the only objects used to describe observability data inside the pipeline. The notebook does not use direct MLflow logging calls.


In [21]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import sys
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import requests

from IPython.display import Markdown, display

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.validation import check_is_fitted

from websensors_flow import (
    MetricRecord,
    PipelineContext,
    PipelineStep,
    StepResult,
    build_pipeline_from_settings,
)
from websensors_flow.config import FlowSettings

WORK_DIR = Path("websensors_flow_genai_notebook_workspace").resolve()
ARTIFACT_DIR = WORK_DIR / "artifacts"
REPORT_DIR = WORK_DIR / "reports"
MLRUNS_DIR = WORK_DIR / "mlruns"

for path in [WORK_DIR, ARTIFACT_DIR, REPORT_DIR, MLRUNS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://raw.githubusercontent.com/rmarcacini/text-collections/refs/heads/master/complete_texts_csvs/Dmoz-Health.csv"
RANDOM_STATE = 42
SESSION_ID = f"dmoz-genai-{uuid.uuid4().hex[:10]}"
USER_ID = "websensors-flow-notebook-user"

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Workspace: {WORK_DIR}")


Python: 3.13.5
Platform: Linux-6.12.88+deb13-rt-amd64-x86_64-with-glibc2.41
Workspace: /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace


## 2. OpenRouter API key

The notebook can run in two modes.

When `use_mock_llm=True`, the notebook simulates the LLM response and no API key is required. This is useful for testing the WebSensors Flow observability path.

When `use_mock_llm=False`, the classifier calls OpenRouter. The API key is read from the `OPENROUTER_API_KEY` environment variable. The key is not stored in the model artifact. The model stores only the name of the environment variable.


In [22]:
api_key_env = "OPENROUTER_API_KEY"

if not os.getenv(api_key_env):
    api_key = getpass("OpenRouter API key, optional when use_mock_llm=True: ").strip()
    if api_key:
        os.environ[api_key_env] = api_key

print(f"API key environment variable configured: {api_key_env} =", "yes" if os.getenv(api_key_env) else "no")


API key environment variable configured: OPENROUTER_API_KEY = yes


## 3. Flow configuration

This configuration follows the same structure used in the scikit-learn tutorial.

The `observability` block configures the backend used by WebSensors Flow. The notebook itself does not call the backend directly.

The pipeline has four steps:

1. `ingest_data` loads a small labeled dataset and emits a dataset record.
2. `preprocess_text` cleans the texts and saves the prepared dataset.
3. `classify_with_llm` creates a scikit-learn-compatible GenAI classifier and predicts the whole batch.
4. `evaluate_and_register_model` evaluates the predictions and emits the final model record.

The key design decision is that row-level predictions are saved as artifacts, not as one metric record per instance. This avoids creating thousands of records when the dataset grows.


In [23]:
flow_config: dict[str, Any] = {
    "project": {
        "name": "websensors-flow-dmoz-health-genai-notebook",
        "version": "1.1.0",
        "description": "DMOZ Health GenAI text classification with aggregated WebSensors Flow observability.",
    },
    "environment": {
        "name": "notebook",
        "owner": "tutorial",
        "tags": {
            "interface": "jupyter",
            "example": "dmoz-health",
            "observability": "genai",
        },
    },
    "runtime": {
        "report_dir": str(REPORT_DIR),
        "fail_fast": True,
        "include_traceback": True,
        "raise_on_failure": False,
        "console": {
            "enabled": True,
            "progress": True,
            "show_metrics": True,
        },
    },
    "observability": {
        "mlflow": {
            "enabled": True,
            "tracking_uri": "http://127.0.0.1:5000",
            "experiment_name": "websensors-flow-dmoz-health-genai-notebook",
            "run_name": "dmoz-health-genai-flow",
            "http_request_timeout": 10,
            "connect_timeout_seconds": 5.0,
        },
        "graylog": {
            "enabled": False,
        },
    },
    "pipeline": {
        "params": {
            "dataset_url": DATASET_URL,
            "random_state": RANDOM_STATE,
            "task": "genai_text_classification",
            "label_column": "class",
            "text_column": "text",
            "session_id": SESSION_ID,
            "user_id": USER_ID,
        },
    },
    "api": {
        "enabled": False,
    },
    "steps": [
        {
            "name": "ingest_data",
            "config": {
                "dataset_url": DATASET_URL,
                "max_classes": 4,
                "examples_per_class": 10,
                "random_state": RANDOM_STATE,
                "sample_artifact_rows": 25,
                "artifact_dir": str(ARTIFACT_DIR / "01_ingest_data"),
            },
        },
        {
            "name": "preprocess_text",
            "config": {
                "min_text_length": 20,
                "artifact_dir": str(ARTIFACT_DIR / "02_preprocess_text"),
            },
        },
        {
            "name": "classify_with_llm",
            "config": {
                "artifact_dir": str(ARTIFACT_DIR / "03_classify_with_llm"),
                "reuse_cached_outputs": True,
                "use_mock_llm": True,
                "registered_model_name": "websensors_flow_dmoz_health_openrouter_classifier",
                "llm": {
                    "provider_name": "openrouter",
                    "base_url": "https://openrouter.ai/api/v1",
                    "api_key_env": "OPENROUTER_API_KEY",
                    "model": "openai/gpt-4o-mini",
                    "temperature": 0.0,
                    "max_tokens": 256,
                    "timeout_seconds": 60,
                    "max_retries": 2,
                    "max_workers": 4,
                },
            },
        },
        {
            "name": "evaluate_and_register_model",
            "config": {
                "artifact_dir": str(ARTIFACT_DIR / "04_evaluate_and_register_model"),
                "registered_model_name": "websensors_flow_dmoz_health_openrouter_classifier",
            },
        },
    ],
}

settings = FlowSettings.model_validate(flow_config).resolve_external_values()
settings


FlowSettings(source_path=None, project=ProjectConfig(name='websensors-flow-dmoz-health-genai-notebook', version='1.1.0', description='DMOZ Health GenAI text classification with aggregated WebSensors Flow observability.'), environment=EnvironmentConfig(name='notebook', deployment_id=None, owner='tutorial', tags={'interface': 'jupyter', 'example': 'dmoz-health', 'observability': 'genai'}), runtime=RuntimeConfig(report_dir='/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/reports', fail_fast=True, include_traceback=True, raise_on_failure=False, console=ConsoleConfig(enabled=True, progress=True, show_metrics=True)), observability=ObservabilityConfig(mlflow=MLflowConfig(enabled=True, tracking_uri='http://127.0.0.1:5000', tracking_uri_env='MLFLOW_TRACKING_URI', experiment_name='websensors-flow-dmoz-health-genai-notebook', experiment_name_env='MLFLOW_EXPERIMENT_NAME', run_name='dmoz-health-genai-flow', artifact_location=None, http_request_t

## 4. Data objects passed between steps

Each step receives the output of the previous step and returns a `StepResult`.

The dataclasses below make the data contract explicit. They also keep large objects, such as dataframes and model artifacts, separate from observability records.


In [24]:
@dataclass
class IngestedDataset:
    dataframe: pd.DataFrame
    dataset_name: str
    source_url: str
    raw_dataset_path: Path
    sample_path: Path
    class_distribution_path: Path


@dataclass
class PreparedTextDataset:
    dataframe: pd.DataFrame
    dataset_name: str
    source_url: str
    text_column: str
    label_column: str
    prepared_dataset_path: Path
    class_distribution_path: Path


@dataclass
class GenAIClassificationResult:
    dataframe: pd.DataFrame
    dataset: PreparedTextDataset
    classifier: Any
    model_name: str
    registered_model_name: str
    provider_name: str
    model_artifact_path: Path
    model_card_path: Path
    system_prompt_path: Path
    prediction_dataset_path: Path
    raw_calls_path: Path
    telemetry_summary_path: Path
    aggregate_usage: dict[str, Any]


@dataclass
class RegisteredGenAIModel:
    model_name: str
    provider_name: str
    model_artifact_path: Path
    model_card_path: Path
    evaluation_dataset_path: Path
    metrics_path: Path
    confusion_matrix_path: Path
    classification_report_path: Path
    metrics: dict[str, float]


## 5. Helper functions

The helper functions save artifacts and normalize text.

Artifacts are files produced by steps. Examples include prepared datasets, prompts, raw LLM calls, predictions, model cards, and serialized models. They are not numeric metrics.


In [25]:
def ensure_dir(path: str | Path) -> Path:
    resolved = Path(path).resolve()
    resolved.mkdir(parents=True, exist_ok=True)
    return resolved


def save_json(data: Any, path: str | Path) -> Path:
    output_path = Path(path).resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return output_path


def save_jsonl(path: str | Path, rows: list[dict[str, Any]]) -> Path:
    output_path = Path(path).resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
    return output_path


def save_dataframe(dataframe: pd.DataFrame, path: str | Path) -> Path:
    output_path = Path(path).resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(output_path, index=False)
    return output_path


def stable_hash(text: str, size: int = 12) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:size]


def clean_text(value: Any) -> str:
    text = str(value or "")
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return default


def show_step_result(title: str, result: StepResult) -> None:
    display(Markdown(f"### {title}"))
    display(Markdown(result.text))
    if result.metrics:
        display(pd.DataFrame([result.metrics]).T.rename(columns={0: "value"}))
    if result.params:
        display(pd.DataFrame([result.params]).T.rename(columns={0: "value"}))


## 6. LLM helper functions

These functions implement the external LLM call and the mock classifier.

The mock mode is intentionally deterministic enough for tutorial testing. The real mode calls OpenRouter and returns a parsed JSON response.

The telemetry collected here is later aggregated by the classifier. The pipeline does not log one metric record per classified row.


In [26]:
def extract_json_object(text: str) -> dict[str, Any]:
    if not text:
        return {}
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except Exception:
        return {}


def post_openrouter_chat(
    *,
    llm_config: dict[str, Any],
    system_prompt: str,
    user_prompt: str,
) -> dict[str, Any]:
    api_key = os.getenv(str(llm_config.get("api_key_env", "OPENROUTER_API_KEY")))
    if not api_key:
        raise RuntimeError("OpenRouter API key is missing.")

    base_url = str(llm_config["base_url"]).rstrip("/")
    endpoint = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": llm_config["model"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": float(llm_config.get("temperature", 0.0)),
        "max_tokens": int(llm_config.get("max_tokens", 256)),
    }

    max_retries = int(llm_config.get("max_retries", 2))
    timeout_seconds = int(llm_config.get("timeout_seconds", 60))
    start = time.time()
    last_error = ""

    for attempt in range(max_retries + 1):
        try:
            response = requests.post(endpoint, headers=headers, json=payload, timeout=timeout_seconds)
            response.raise_for_status()
            data = response.json()
            content = data["choices"][0]["message"]["content"]
            usage = data.get("usage") or {}
            return {
                "success": True,
                "attempts": attempt + 1,
                "latency_seconds": time.time() - start,
                "raw_response": content,
                "parsed_response": extract_json_object(content),
                "usage": usage,
                "error": "",
            }
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(min(2 ** attempt, 5))

    return {
        "success": False,
        "attempts": max_retries + 1,
        "latency_seconds": time.time() - start,
        "raw_response": "",
        "parsed_response": {},
        "usage": {},
        "error": last_error,
    }


def mock_llm_classify(text: str, classes: list[str]) -> dict[str, Any]:
    lowered = text.lower()
    scores = {class_name: 0 for class_name in classes}

    keyword_map = {
        "fitness": ["fitness", "exercise", "training", "workout", "sport", "running"],
        "nutrition": ["nutrition", "diet", "food", "vitamin", "meal", "calorie"],
        "medical": ["medical", "hospital", "patient", "disease", "treatment", "diagnosis"],
        "pharmacy": ["pharmacy", "drug", "medicine", "medication", "prescription"],
        "mental_health": ["mental", "psychology", "stress", "anxiety", "therapy"],
    }

    for class_name in classes:
        terms = keyword_map.get(class_name.lower(), [class_name.lower()])
        scores[class_name] = sum(term in lowered for term in terms)

    predicted = max(scores, key=scores.get)
    if scores[predicted] == 0:
        predicted = classes[int(stable_hash(text, 8), 16) % len(classes)]

    return {
        "predicted_class": predicted,
        "confidence": 0.75 if scores[predicted] > 0 else 0.45,
        "reasoning": "Mock classifier selected the class using simple keyword evidence.",
        "evidence": [predicted],
    }


def run_parallel(items: list[dict[str, Any]], worker_fn, *, max_workers: int) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(worker_fn, item): item for item in items}
        for future in as_completed(futures):
            results.append(future.result())
    return sorted(results, key=lambda row: row["row_index"])


## 7. Scikit-learn-compatible GenAI classifier

This class is the main adaptation for the GenAI case.

The external LLM is not a trained scikit-learn estimator, but we wrap it with the familiar `fit` and `predict` interface:

- `fit(X, y)` stores the class list and builds the system prompt.
- `predict(X)` classifies a batch of texts using either OpenRouter or the mock classifier.
- `last_prediction_details_` stores row-level predictions for artifacts.
- `last_telemetry_summary_` stores aggregated GenAI metrics for observability.

This wrapper gives the pipeline a real model object to serialize with `joblib`. The serialized artifact is then passed to WebSensors Flow through `StepResult.artifacts` and a `MetricRecord` named `final_model`.


In [27]:
def build_classifier_system_prompt(classes: list[str]) -> str:
    class_lines = "\n".join(f"- {class_name}" for class_name in classes)
    return f"""
You are a strict text classifier for health-related web pages.

Choose exactly one class from the allowed list:
{class_lines}

Return only a JSON object with this schema:
{{
  "predicted_class": "one allowed class",
  "confidence": 0.0,
  "reasoning": "short explanation",
  "evidence": ["short evidence from the text"]
}}
""".strip()


def build_classifier_user_prompt(text: str) -> str:
    return f"Classify the following web page text.\n\nTEXT:\n{text}"


class OpenRouterGenAIClassifier(BaseEstimator, ClassifierMixin):
    """A scikit-learn-compatible wrapper around an external GenAI chat classifier."""

    def __init__(
        self,
        *,
        model_name: str = "openai/gpt-4o-mini",
        provider_name: str = "openrouter",
        base_url: str = "https://openrouter.ai/api/v1",
        api_key_env: str = "OPENROUTER_API_KEY",
        temperature: float = 0.0,
        max_tokens: int = 256,
        timeout_seconds: int = 60,
        max_retries: int = 2,
        max_workers: int = 4,
        use_mock_llm: bool = True,
    ):
        self.model_name = model_name
        self.provider_name = provider_name
        self.base_url = base_url
        self.api_key_env = api_key_env
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.timeout_seconds = timeout_seconds
        self.max_retries = max_retries
        self.max_workers = max_workers
        self.use_mock_llm = use_mock_llm

    def fit(self, X, y):
        labels = pd.Series(y).dropna().astype(str)
        self.classes_ = np.array(sorted(labels.unique().tolist()))
        self.system_prompt_ = build_classifier_system_prompt(self.classes_.tolist())
        self.is_fitted_ = True
        self.last_prediction_details_ = []
        self.last_raw_calls_ = []
        self.last_telemetry_summary_ = {}
        return self

    def _llm_config(self) -> dict[str, Any]:
        return {
            "provider_name": self.provider_name,
            "base_url": self.base_url,
            "api_key_env": self.api_key_env,
            "model": self.model_name,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "timeout_seconds": self.timeout_seconds,
            "max_retries": self.max_retries,
        }

    def _predict_one(self, row: dict[str, Any]) -> dict[str, Any]:
        text = str(row["text"])
        user_prompt = build_classifier_user_prompt(text)
        start = time.time()

        if self.use_mock_llm:
            parsed = mock_llm_classify(text, self.classes_.tolist())
            prompt_tokens = max(1, len((self.system_prompt_ + " " + user_prompt).split()))
            completion_tokens = max(1, len(json.dumps(parsed).split()))
            latency_seconds = time.time() - start
            raw_call = {
                "success": True,
                "attempts": 1,
                "latency_seconds": latency_seconds,
                "raw_response": json.dumps(parsed, ensure_ascii=False),
                "parsed_response": parsed,
                "usage": {
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": completion_tokens,
                    "total_tokens": prompt_tokens + completion_tokens,
                },
                "error": "",
            }
        else:
            raw_call = post_openrouter_chat(
                llm_config=self._llm_config(),
                system_prompt=self.system_prompt_,
                user_prompt=user_prompt,
            )
            parsed = raw_call.get("parsed_response") or {}

        predicted_class = str(parsed.get("predicted_class", "")).strip()
        if predicted_class not in self.classes_:
            predicted_class = self.classes_.tolist()[0]

        usage = raw_call.get("usage") or {}
        prompt_tokens = int(usage.get("prompt_tokens") or usage.get("input_tokens") or 0)
        completion_tokens = int(usage.get("completion_tokens") or usage.get("output_tokens") or 0)
        total_tokens = int(usage.get("total_tokens") or prompt_tokens + completion_tokens)
        latency_seconds = safe_float(raw_call.get("latency_seconds"), 0.0)

        return {
            "row_index": int(row["row_index"]),
            "predicted_class": predicted_class,
            "confidence": safe_float(parsed.get("confidence"), 0.0),
            "reasoning": str(parsed.get("reasoning", "")),
            "evidence": parsed.get("evidence", []),
            "success": bool(raw_call.get("success", False)),
            "attempts": int(raw_call.get("attempts", 0)),
            "latency_seconds": latency_seconds,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "error": str(raw_call.get("error", "")),
            "raw_response": raw_call.get("raw_response", ""),
        }

    def predict(self, X):
        check_is_fitted(self, "is_fitted_")
        texts = pd.Series(X).astype(str).tolist()
        items = [{"row_index": index, "text": text} for index, text in enumerate(texts)]
        start = time.time()
        details = run_parallel(items, self._predict_one, max_workers=int(self.max_workers))
        batch_latency = time.time() - start

        total_prompt_tokens = int(sum(row["prompt_tokens"] for row in details))
        total_completion_tokens = int(sum(row["completion_tokens"] for row in details))
        total_tokens = int(sum(row["total_tokens"] for row in details))
        request_count = len(details)
        successful_requests = int(sum(1 for row in details if row["success"]))
        failed_requests = request_count - successful_requests
        latencies = [float(row["latency_seconds"]) for row in details]

        self.last_prediction_details_ = details
        self.last_raw_calls_ = [
            {
                "row_index": row["row_index"],
                "success": row["success"],
                "attempts": row["attempts"],
                "latency_seconds": row["latency_seconds"],
                "prompt_tokens": row["prompt_tokens"],
                "completion_tokens": row["completion_tokens"],
                "total_tokens": row["total_tokens"],
                "raw_response": row["raw_response"],
                "error": row["error"],
            }
            for row in details
        ]
        self.last_telemetry_summary_ = {
            "gen_ai_request_count": request_count,
            "gen_ai_successful_requests": successful_requests,
            "gen_ai_failed_requests": failed_requests,
            "gen_ai_total_latency_seconds": float(batch_latency),
            "gen_ai_mean_latency_seconds": float(np.mean(latencies)) if latencies else 0.0,
            "gen_ai_p95_latency_seconds": float(np.percentile(latencies, 95)) if latencies else 0.0,
            "gen_ai_input_tokens": total_prompt_tokens,
            "gen_ai_output_tokens": total_completion_tokens,
            "gen_ai_total_tokens": total_tokens,
            "gen_ai_tokens_per_second": float(total_tokens / batch_latency) if batch_latency > 0 else 0.0,
            "gen_ai_mean_input_tokens_per_request": float(total_prompt_tokens / request_count) if request_count else 0.0,
            "gen_ai_mean_output_tokens_per_request": float(total_completion_tokens / request_count) if request_count else 0.0,
            "gen_ai_mean_total_tokens_per_request": float(total_tokens / request_count) if request_count else 0.0,
        }
        return np.array([row["predicted_class"] for row in details])


## 8. Step 1: ingest dataset

The ingestion step loads the data and emits a single dataset record.

This is important for observability because the dataset should appear as one dataset-level object, not as one record per row.


In [28]:
class IngestDataStep(PipelineStep):
    """Load DMOZ Health and expose one aggregated dataset record."""

    name = "ingest_data"

    def execute(self, input: Any, context: PipelineContext) -> StepResult:
        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        dataset_url = config["dataset_url"]
        max_classes = int(config.get("max_classes", 4))
        examples_per_class = int(config.get("examples_per_class", 10))
        random_state = int(config.get("random_state", RANDOM_STATE))
        sample_artifact_rows = int(config.get("sample_artifact_rows", 25))

        try:
            dataframe = pd.read_csv(dataset_url)
        except Exception as exc:
            context.log.warning(
                "The remote dataset could not be loaded. A small local fallback dataset will be used.",
                metadata={"error_type": type(exc).__name__, "error_message": str(exc)},
            )
            dataframe = pd.DataFrame(
                {
                    "file_name": [f"doc_{i:03d}" for i in range(24)],
                    "text": [
                        "Medical information about symptoms treatment and patient care in hospitals.",
                        "Healthcare professionals discuss disease prevention and clinical diagnosis.",
                        "A hospital guide explains treatment options and medical appointments.",
                        "Patient education material about chronic disease and health services.",
                        "Fitness training plans improve strength flexibility and exercise routines.",
                        "Workout programs include aerobic exercise running and muscle training.",
                        "A guide to physical activity fitness goals and healthy movement.",
                        "Exercise routines help endurance strength and sports conditioning.",
                        "Nutrition resources explain vitamins minerals diets and healthy meals.",
                        "A diet guide covers food calories protein and balanced nutrition.",
                        "Healthy eating article about fruits vegetables and meal planning.",
                        "Nutrition facts describe dietary choices weight control and food groups.",
                        "Pharmacy pages describe prescriptions medication dosage and drug safety.",
                        "Medication guides explain side effects and pharmacy services.",
                        "Drug information includes interactions prescriptions and safe use.",
                        "Pharmacy resources cover medicine dosage and prescription refills.",
                        "Mental health resources discuss therapy stress anxiety and support.",
                        "Psychology articles explain stress management and emotional health.",
                        "Therapy information covers anxiety depression and counseling.",
                        "Mental wellness pages describe counseling and support services.",
                        "Clinical resources describe physicians medicine and public health.",
                        "Training advice for gym workouts stretching and physical performance.",
                        "Dietary recommendations include protein fiber vitamins and hydration.",
                        "Prescription safety article about medication and patient guidance.",
                    ],
                    "class": ["medical"] * 4 + ["fitness"] * 4 + ["nutrition"] * 4 + ["pharmacy"] * 4 + ["mental_health"] * 4 + ["medical", "fitness", "nutrition", "pharmacy"],
                }
            )

        expected_columns = {"file_name", "text", "class"}
        missing_columns = expected_columns.difference(dataframe.columns)
        if missing_columns:
            raise ValueError(f"The dataset is missing required columns: {sorted(missing_columns)}")

        dataframe = dataframe.dropna(subset=["text", "class"]).copy()
        dataframe["text"] = dataframe["text"].astype(str)
        dataframe["class"] = dataframe["class"].astype(str)

        selected_classes = dataframe["class"].value_counts().head(max_classes).index.tolist()
        dataframe = dataframe[dataframe["class"].isin(selected_classes)].copy()
        dataframe = (
            dataframe.groupby("class", group_keys=False)
            .apply(lambda group: group.sample(min(len(group), examples_per_class), random_state=random_state))
            .sample(frac=1.0, random_state=random_state)
            .reset_index(drop=True)
        )
        dataframe["row_id"] = [f"dmoz_{i:04d}" for i in range(len(dataframe))]

        raw_dataset_path = save_dataframe(dataframe, artifact_dir / "raw_dataset.csv")
        sample_path = save_dataframe(dataframe.head(sample_artifact_rows), artifact_dir / "dataset_sample.csv")
        class_distribution = dataframe["class"].value_counts().rename_axis("class").reset_index(name="count")
        class_distribution_path = save_dataframe(class_distribution, artifact_dir / "class_distribution.csv")

        class_counts = dataframe["class"].value_counts()
        output = IngestedDataset(
            dataframe=dataframe,
            dataset_name="Dmoz-Health",
            source_url=dataset_url,
            raw_dataset_path=raw_dataset_path,
            sample_path=sample_path,
            class_distribution_path=class_distribution_path,
        )

        dataset_record = MetricRecord(
            name="dataset",
            params={
                "dataset_name": output.dataset_name,
                "source": dataset_url,
                "rows": len(dataframe),
                "text_column": "text",
                "label_column": "class",
                "task": "genai_text_classification",
            },
            metrics={
                "dataset_rows": len(dataframe),
                "dataset_classes": int(class_counts.shape[0]),
                "dataset_min_class_count": int(class_counts.min()) if not class_counts.empty else 0,
                "dataset_max_class_count": int(class_counts.max()) if not class_counts.empty else 0,
            },
            metadata={
                "context": "evaluation",
                "format": "csv",
                "row_level_records_logged": False,
            },
            artifacts={
                "dataset_sample": str(sample_path),
                "raw_dataset": str(raw_dataset_path),
                "class_distribution": str(class_distribution_path),
            },
        )

        return StepResult(
            output=output,
            has_output=True,
            text=f"Loaded {len(dataframe)} documents from {output.dataset_name} across {class_counts.shape[0]} classes.",
            metrics={
                "rows": len(dataframe),
                "classes": int(class_counts.shape[0]),
                "min_class_count": int(class_counts.min()) if not class_counts.empty else 0,
                "max_class_count": int(class_counts.max()) if not class_counts.empty else 0,
            },
            params={
                "dataset_url": dataset_url,
                "max_classes": max_classes,
                "examples_per_class": examples_per_class,
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "columns": list(dataframe.columns),
            },
            artifacts={
                "raw_dataset": str(raw_dataset_path),
                "dataset_sample": str(sample_path),
                "class_distribution": str(class_distribution_path),
            },
            metric_records=[dataset_record],
        )


### Run the ingestion step directly

Running each step manually is useful in a tutorial because it shows what the step returns before the complete pipeline is executed.


In [29]:
manual_context = PipelineContext(settings=settings, run_id="manual-notebook-run", pipeline_name=settings.project.name)
ingest_step = IngestDataStep()
manual_context.set_current_step(ingest_step.step_name, 1)
ingested = ingest_step.execute(None, manual_context)
show_step_result("Ingestion step", ingested)
ingested.output.dataframe.head()


/tmp/ipykernel_33852/3574501119.py:68: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(min(len(group), examples_per_class), random_state=random_state))


### Ingestion step

Loaded 40 documents from Dmoz-Health across 4 classes.

,value
rows,40
classes,4
min_class_count,10
max_class_count,10


,value
dataset_url,https://raw.githubusercontent.com/rmarcacini/t...
max_classes,4
examples_per_class,10


,file_name,text,class,row_id
0,1580448.txt,"Biotherapy Clinic Holistic medicine, acupunctu...",Alternative,dmoz_0000
1,1581702.txt,Ireland's Holistic Directory Listings of holis...,Alternative,dmoz_0001
2,1580794.txt,Excel With Ease Coaching Dr. Rachna D. Jain is...,Alternative,dmoz_0002
3,1586826.txt,Bright Cross Animal Clinic Community veterinar...,Animal,dmoz_0003
4,1578126.txt,Matrix Institute on Addictions Seeks to improv...,Addictions,dmoz_0004


## 9. Step 2: preprocess text

This step creates the text column that will be passed to the GenAI classifier.

The output is still a normal dataframe artifact. The step records only aggregated metrics such as rows before and after filtering.


In [30]:
class PreprocessTextStep(PipelineStep):
    """Clean text while preserving the original row id and label."""

    name = "preprocess_text"

    def execute(self, input: IngestedDataset, context: PipelineContext) -> StepResult:
        if not isinstance(input, IngestedDataset):
            raise TypeError("PreprocessTextStep expects an IngestedDataset object.")

        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        min_text_length = int(config.get("min_text_length", 20))

        dataframe = input.dataframe[["row_id", "text", "class"]].copy()
        rows_before = len(dataframe)
        dataframe["text"] = dataframe["text"].map(clean_text)
        dataframe = dataframe[dataframe["text"].str.len() >= min_text_length].reset_index(drop=True)
        rows_after = len(dataframe)

        prepared_dataset_path = save_dataframe(dataframe, artifact_dir / "prepared_dataset.csv")
        class_distribution = dataframe["class"].value_counts().rename_axis("class").reset_index(name="count")
        class_distribution_path = save_dataframe(class_distribution, artifact_dir / "class_distribution.csv")

        output = PreparedTextDataset(
            dataframe=dataframe,
            dataset_name=input.dataset_name,
            source_url=input.source_url,
            text_column="text",
            label_column="class",
            prepared_dataset_path=prepared_dataset_path,
            class_distribution_path=class_distribution_path,
        )

        return StepResult(
            output=output,
            has_output=True,
            text=f"Prepared {rows_after} documents for GenAI classification.",
            metrics={
                "rows_before_filter": rows_before,
                "rows_after_filter": rows_after,
                "removed_rows": rows_before - rows_after,
                "classes_after_filter": int(dataframe["class"].nunique()),
            },
            params={
                "min_text_length": min_text_length,
                "text_column": "text",
                "label_column": "class",
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "row_level_records_logged": False,
            },
            artifacts={
                "prepared_dataset": str(prepared_dataset_path),
                "class_distribution": str(class_distribution_path),
            },
        )


### Run the preprocessing step directly


In [31]:
preprocess_step = PreprocessTextStep()
manual_context.set_current_step(preprocess_step.step_name, 2)
prepared = preprocess_step.execute(ingested.output, manual_context)
show_step_result("Preprocessing step", prepared)
prepared.output.dataframe.head()


### Preprocessing step

Prepared 40 documents for GenAI classification.

,value
rows_before_filter,40
rows_after_filter,40
removed_rows,0
classes_after_filter,4


,value
min_text_length,20
text_column,text
label_column,class


,row_id,text,class
0,dmoz_0000,"Biotherapy Clinic Holistic medicine, acupunctu...",Alternative
1,dmoz_0001,Ireland's Holistic Directory Listings of holis...,Alternative
2,dmoz_0002,Excel With Ease Coaching Dr. Rachna D. Jain is...,Alternative
3,dmoz_0003,Bright Cross Animal Clinic Community veterinar...,Animal
4,dmoz_0004,Matrix Institute on Addictions Seeks to improv...,Addictions


## 10. Step 3: classify with the GenAI model

This step creates the `OpenRouterGenAIClassifier`, calls `fit`, and then calls `predict` over the complete prepared dataset.

The classifier is serialized with `joblib`, so the pipeline has a concrete model artifact. The step also saves:

- the system prompt;
- the full prediction table;
- raw LLM call details;
- an aggregated GenAI telemetry summary.

The step emits one model candidate record. It does not emit one metric record per classified row.


In [32]:
class ClassifyWithLLMStep(PipelineStep):
    """Create a scikit-learn-compatible GenAI classifier and classify the prepared dataset."""

    name = "classify_with_llm"

    def execute(self, input: PreparedTextDataset, context: PipelineContext) -> StepResult:
        if not isinstance(input, PreparedTextDataset):
            raise TypeError("ClassifyWithLLMStep expects a PreparedTextDataset object.")

        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        llm_config = dict(config.get("llm", {}))
        use_mock_llm = bool(config.get("use_mock_llm", True))
        registered_model_name = str(config.get("registered_model_name", llm_config.get("model", "genai_classifier")))

        dataframe = input.dataframe.copy()
        X = dataframe[input.text_column]
        y = dataframe[input.label_column]

        classifier = OpenRouterGenAIClassifier(
            model_name=str(llm_config.get("model", "openai/gpt-4o-mini")),
            provider_name=str(llm_config.get("provider_name", "openrouter")),
            base_url=str(llm_config.get("base_url", "https://openrouter.ai/api/v1")),
            api_key_env=str(llm_config.get("api_key_env", "OPENROUTER_API_KEY")),
            temperature=float(llm_config.get("temperature", 0.0)),
            max_tokens=int(llm_config.get("max_tokens", 256)),
            timeout_seconds=int(llm_config.get("timeout_seconds", 60)),
            max_retries=int(llm_config.get("max_retries", 2)),
            max_workers=int(llm_config.get("max_workers", 4)),
            use_mock_llm=use_mock_llm,
        )

        classifier.fit(X, y)
        predictions = classifier.predict(X)

        dataframe["predicted_class"] = predictions
        details_by_row = {row["row_index"]: row for row in classifier.last_prediction_details_}
        dataframe["confidence"] = [details_by_row[index]["confidence"] for index in range(len(dataframe))]
        dataframe["llm_success"] = [details_by_row[index]["success"] for index in range(len(dataframe))]
        dataframe["llm_latency_seconds"] = [details_by_row[index]["latency_seconds"] for index in range(len(dataframe))]
        dataframe["llm_total_tokens"] = [details_by_row[index]["total_tokens"] for index in range(len(dataframe))]
        dataframe["llm_reasoning"] = [details_by_row[index]["reasoning"] for index in range(len(dataframe))]

        system_prompt_path = artifact_dir / "classifier_system_prompt.txt"
        system_prompt_path.write_text(classifier.system_prompt_, encoding="utf-8")
        prediction_dataset_path = save_dataframe(dataframe, artifact_dir / "predictions.csv")
        raw_calls_path = save_jsonl(artifact_dir / "raw_llm_calls.jsonl", classifier.last_raw_calls_)
        telemetry_summary_path = save_json(classifier.last_telemetry_summary_, artifact_dir / "genai_telemetry_summary.json")

        model_artifact_path = artifact_dir / "genai_classifier_sklearn_wrapper.joblib"
        joblib.dump(
            {
                "model": classifier,
                "registered_model_name": registered_model_name,
                "model_type": "sklearn_compatible_genai_classifier",
                "classes": classifier.classes_.tolist(),
                "provider_name": classifier.provider_name,
                "external_model_name": classifier.model_name,
                "api_key_env": classifier.api_key_env,
            },
            model_artifact_path,
        )

        model_card = {
            "registered_model_name": registered_model_name,
            "model_type": "sklearn_compatible_genai_classifier",
            "model_interface": ["fit", "predict"],
            "external_model_name": classifier.model_name,
            "provider_name": classifier.provider_name,
            "task": "genai_text_classification",
            "classes": classifier.classes_.tolist(),
            "api_key_storage": "environment_variable_name_only",
            "api_key_env": classifier.api_key_env,
            "mock_mode": classifier.use_mock_llm,
            "row_level_metric_records_logged": False,
            "row_level_details_saved_as_artifacts": True,
        }
        model_card_path = save_json(model_card, artifact_dir / "genai_model_card.json")

        telemetry = classifier.last_telemetry_summary_
        model_candidate_record = MetricRecord(
            name="model_candidate",
            params={
                "model_name": registered_model_name,
                "model_family": "sklearn_compatible_genai_classifier",
                "external_model_name": classifier.model_name,
                "provider_name": classifier.provider_name,
                "gen_ai.provider.name": classifier.provider_name,
                "gen_ai.request.model": classifier.model_name,
                "gen_ai.operation.name": "chat",
                "use_mock_llm": str(classifier.use_mock_llm),
            },
            metrics={
                "prediction_rows": len(dataframe),
                "gen_ai_request_count": int(telemetry.get("gen_ai_request_count", 0)),
                "gen_ai_successful_requests": int(telemetry.get("gen_ai_successful_requests", 0)),
                "gen_ai_failed_requests": int(telemetry.get("gen_ai_failed_requests", 0)),
                "gen_ai_total_latency_seconds": float(telemetry.get("gen_ai_total_latency_seconds", 0.0)),
                "gen_ai_mean_latency_seconds": float(telemetry.get("gen_ai_mean_latency_seconds", 0.0)),
                "gen_ai_p95_latency_seconds": float(telemetry.get("gen_ai_p95_latency_seconds", 0.0)),
                "gen_ai_input_tokens": int(telemetry.get("gen_ai_input_tokens", 0)),
                "gen_ai_output_tokens": int(telemetry.get("gen_ai_output_tokens", 0)),
                "gen_ai_total_tokens": int(telemetry.get("gen_ai_total_tokens", 0)),
                "gen_ai_tokens_per_second": float(telemetry.get("gen_ai_tokens_per_second", 0.0)),
            },
            metadata={
                "artifact": str(model_artifact_path),
                "model_card": str(model_card_path),
                "row_level_records_logged": False,
            },
            artifacts={
                "model_artifact": str(model_artifact_path),
                "model_card": str(model_card_path),
                "classifier_system_prompt": str(system_prompt_path),
                "prediction_dataset": str(prediction_dataset_path),
                "raw_llm_calls": str(raw_calls_path),
                "genai_telemetry_summary": str(telemetry_summary_path),
            },
        )

        output = GenAIClassificationResult(
            dataframe=dataframe,
            dataset=input,
            classifier=classifier,
            model_name=classifier.model_name,
            registered_model_name=registered_model_name,
            provider_name=classifier.provider_name,
            model_artifact_path=model_artifact_path,
            model_card_path=model_card_path,
            system_prompt_path=system_prompt_path,
            prediction_dataset_path=prediction_dataset_path,
            raw_calls_path=raw_calls_path,
            telemetry_summary_path=telemetry_summary_path,
            aggregate_usage=telemetry,
        )

        return StepResult(
            output=output,
            has_output=True,
            text=f"Classified {len(dataframe)} rows with the GenAI classifier `{registered_model_name}`.",
            metrics={
                "prediction_rows": len(dataframe),
                **telemetry,
            },
            params={
                "registered_model_name": registered_model_name,
                "external_model_name": classifier.model_name,
                "provider_name": classifier.provider_name,
                "model_family": "sklearn_compatible_genai_classifier",
                "use_mock_llm": classifier.use_mock_llm,
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "row_level_metric_records_logged": False,
                "row_level_details_saved_as_artifacts": True,
            },
            artifacts={
                "model_artifact": str(model_artifact_path),
                "model_card": str(model_card_path),
                "classifier_system_prompt": str(system_prompt_path),
                "prediction_dataset": str(prediction_dataset_path),
                "raw_llm_calls": str(raw_calls_path),
                "genai_telemetry_summary": str(telemetry_summary_path),
            },
            metric_records=[model_candidate_record],
        )


### Run the GenAI classification step directly


In [33]:
classifier_step = ClassifyWithLLMStep()
manual_context.set_current_step(classifier_step.step_name, 3)
classified = classifier_step.execute(prepared.output, manual_context)
show_step_result("GenAI classification step", classified)
classified.output.dataframe[["row_id", "class", "predicted_class", "confidence", "llm_total_tokens"]].head()


### GenAI classification step

Classified 40 rows with the GenAI classifier `websensors_flow_dmoz_health_openrouter_classifier`.

,value
prediction_rows,4.000000e+01
gen_ai_request_count,4.000000e+01
gen_ai_successful_requests,4.000000e+01
gen_ai_failed_requests,0.000000e+00
gen_ai_total_latency_seconds,3.712654e-03
gen_ai_mean_latency_seconds,2.667904e-05
gen_ai_p95_latency_seconds,5.081892e-05
gen_ai_input_tokens,3.130000e+03
gen_ai_output_tokens,6.400000e+02
gen_ai_total_tokens,3.770000e+03


,value
registered_model_name,websensors_flow_dmoz_health_openrouter_classifier
external_model_name,openai/gpt-4o-mini
provider_name,openrouter
model_family,sklearn_compatible_genai_classifier
use_mock_llm,True


,row_id,class,predicted_class,confidence,llm_total_tokens
0,dmoz_0000,Alternative,Conditions,0.45,98
1,dmoz_0001,Alternative,Addictions,0.45,93
2,dmoz_0002,Alternative,Conditions,0.45,105
3,dmoz_0003,Animal,Animal,0.75,93
4,dmoz_0004,Addictions,Addictions,0.75,105


## 11. Step 4: evaluate and register the GenAI model record

This step evaluates the batch predictions and emits the final model observability record.

The model record points to the serialized scikit-learn-compatible GenAI classifier artifact. It also links the dataset, prompt, predictions, raw calls, telemetry summary, classification report, and confusion matrix.

GenAI metrics are aggregated at model level:

- request count;
- successful and failed requests;
- total, mean, and p95 latency;
- input, output, and total tokens;
- tokens per second;
- mean tokens per request;
- classification metrics such as accuracy and macro F1.

The row-level classification details remain in artifacts only.


In [34]:
class EvaluateAndRegisterGenAIModelStep(PipelineStep):
    """Evaluate the batch output and emit one final model record."""

    name = "evaluate_and_register_model"

    def execute(self, input: GenAIClassificationResult, context: PipelineContext) -> StepResult:
        if not isinstance(input, GenAIClassificationResult):
            raise TypeError("EvaluateAndRegisterGenAIModelStep expects a GenAIClassificationResult object.")

        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        registered_model_name = str(config.get("registered_model_name", input.registered_model_name))

        dataframe = input.dataframe.copy()
        label_column = input.dataset.label_column
        classes = sorted(dataframe[label_column].unique().tolist())

        accuracy = float(accuracy_score(dataframe[label_column], dataframe["predicted_class"]))
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            dataframe[label_column], dataframe["predicted_class"], average="macro", zero_division=0
        )
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
            dataframe[label_column], dataframe["predicted_class"], average="weighted", zero_division=0
        )

        report_dict = classification_report(
            dataframe[label_column],
            dataframe["predicted_class"],
            output_dict=True,
            zero_division=0,
        )

        confusion = pd.DataFrame(
            confusion_matrix(dataframe[label_column], dataframe["predicted_class"], labels=classes),
            index=classes,
            columns=classes,
        )

        evaluation_dataset_path = save_dataframe(dataframe, artifact_dir / "final_evaluation_dataset.csv")
        confusion_matrix_path = artifact_dir / "confusion_matrix.csv"
        confusion.to_csv(confusion_matrix_path)
        classification_report_path = save_json(report_dict, artifact_dir / "classification_report.json")

        final_metrics = {
            "accuracy": accuracy,
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "f1_macro": float(f1_macro),
            "precision_weighted": float(precision_weighted),
            "recall_weighted": float(recall_weighted),
            "f1_weighted": float(f1_weighted),
        }
        genai_metrics = {
            "gen_ai_request_count": int(input.aggregate_usage.get("gen_ai_request_count", 0)),
            "gen_ai_successful_requests": int(input.aggregate_usage.get("gen_ai_successful_requests", 0)),
            "gen_ai_failed_requests": int(input.aggregate_usage.get("gen_ai_failed_requests", 0)),
            "gen_ai_total_latency_seconds": float(input.aggregate_usage.get("gen_ai_total_latency_seconds", 0.0)),
            "gen_ai_mean_latency_seconds": float(input.aggregate_usage.get("gen_ai_mean_latency_seconds", 0.0)),
            "gen_ai_p95_latency_seconds": float(input.aggregate_usage.get("gen_ai_p95_latency_seconds", 0.0)),
            "gen_ai_input_tokens": int(input.aggregate_usage.get("gen_ai_input_tokens", 0)),
            "gen_ai_output_tokens": int(input.aggregate_usage.get("gen_ai_output_tokens", 0)),
            "gen_ai_total_tokens": int(input.aggregate_usage.get("gen_ai_total_tokens", 0)),
            "gen_ai_tokens_per_second": float(input.aggregate_usage.get("gen_ai_tokens_per_second", 0.0)),
            "gen_ai_mean_input_tokens_per_request": float(input.aggregate_usage.get("gen_ai_mean_input_tokens_per_request", 0.0)),
            "gen_ai_mean_output_tokens_per_request": float(input.aggregate_usage.get("gen_ai_mean_output_tokens_per_request", 0.0)),
            "gen_ai_mean_total_tokens_per_request": float(input.aggregate_usage.get("gen_ai_mean_total_tokens_per_request", 0.0)),
        }

        metrics_path = save_json({**final_metrics, **genai_metrics}, artifact_dir / "final_metrics.json")

        model_card = {
            "registered_model_name": registered_model_name,
            "model_type": "sklearn_compatible_genai_classifier",
            "model_interface": ["fit", "predict"],
            "external_model_name": input.model_name,
            "provider_name": input.provider_name,
            "task": "genai_text_classification",
            "dataset_name": input.dataset.dataset_name,
            "dataset_source": input.dataset.source_url,
            "classes": classes,
            "model_artifact": str(input.model_artifact_path),
            "prompt_artifact": str(input.system_prompt_path),
            "prediction_artifact": str(input.prediction_dataset_path),
            "raw_calls_artifact": str(input.raw_calls_path),
            "evaluation_artifact": str(evaluation_dataset_path),
            "classification_metrics": final_metrics,
            "genai_metrics": genai_metrics,
            "observability": {
                "dataset_record": "dataset",
                "model_record": "final_model",
                "row_level_metric_records_logged": False,
                "row_level_details_saved_as_artifacts": True,
                "backend_called_directly_by_notebook": False,
            },
        }
        model_card_path = save_json(model_card, artifact_dir / "final_genai_model_card.json")

        class_records: list[MetricRecord] = []
        for class_name, values in report_dict.items():
            if not isinstance(values, dict) or class_name in {"accuracy", "macro avg", "weighted avg"}:
                continue
            class_records.append(
                MetricRecord(
                    name="class_metrics",
                    params={
                        "class_name": class_name,
                        "model_name": registered_model_name,
                    },
                    metrics={
                        "precision": float(values.get("precision", 0.0)),
                        "recall": float(values.get("recall", 0.0)),
                        "f1_score": float(values.get("f1-score", 0.0)),
                        "support": float(values.get("support", 0.0)),
                    },
                    metadata={
                        "evaluation_split": "batch",
                        "dataset_name": input.dataset.dataset_name,
                    },
                )
            )

        final_model_record = MetricRecord(
            name="final_model",
            params={
                "model_name": registered_model_name,
                "model_family": "sklearn_compatible_genai_classifier",
                "model_interface": "fit_predict",
                "external_model_name": input.model_name,
                "provider_name": input.provider_name,
                "task": "genai_text_classification",
                "dataset_name": input.dataset.dataset_name,
                "gen_ai.provider.name": input.provider_name,
                "gen_ai.request.model": input.model_name,
                "gen_ai.operation.name": "chat",
            },
            metrics={
                **final_metrics,
                **genai_metrics,
            },
            metadata={
                "artifact": str(input.model_artifact_path),
                "model_type": "sklearn_compatible_genai_classifier",
                "row_level_records_logged": False,
                "backend_called_directly_by_notebook": False,
            },
            artifacts={
                "model_artifact": str(input.model_artifact_path),
                "model_card": str(model_card_path),
                "evaluation_dataset": str(evaluation_dataset_path),
                "classification_report": str(classification_report_path),
                "confusion_matrix": str(confusion_matrix_path),
                "metrics": str(metrics_path),
                "classifier_prompt": str(input.system_prompt_path),
                "prediction_dataset": str(input.prediction_dataset_path),
                "raw_calls": str(input.raw_calls_path),
                "genai_telemetry_summary": str(input.telemetry_summary_path),
            },
        )

        output = RegisteredGenAIModel(
            model_name=registered_model_name,
            provider_name=input.provider_name,
            model_artifact_path=input.model_artifact_path,
            model_card_path=model_card_path,
            evaluation_dataset_path=evaluation_dataset_path,
            metrics_path=metrics_path,
            confusion_matrix_path=confusion_matrix_path,
            classification_report_path=classification_report_path,
            metrics=final_metrics,
        )

        return StepResult(
            output=output,
            has_output=True,
            text=f"Registered GenAI model record `{registered_model_name}` with macro F1 {float(f1_macro):.3f}.",
            metrics={
                "rows": len(dataframe),
                "classes": len(classes),
                **final_metrics,
                **genai_metrics,
            },
            params={
                "registered_model_name": registered_model_name,
                "external_model_name": input.model_name,
                "provider_name": input.provider_name,
                "model_family": "sklearn_compatible_genai_classifier",
                "task": "genai_text_classification",
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "dataset_name": input.dataset.dataset_name,
                "model_artifact": str(input.model_artifact_path),
                "row_level_metric_records_logged": False,
            },
            artifacts={
                "model_bundle": str(input.model_artifact_path),
                "model_card": str(model_card_path),
                "evaluation_dataset": str(evaluation_dataset_path),
                "classification_report": str(classification_report_path),
                "confusion_matrix": str(confusion_matrix_path),
                "metrics": str(metrics_path),
            },
            metric_records=[final_model_record] + class_records,
        )


### Run the evaluation and model-record step directly


In [35]:
register_step = EvaluateAndRegisterGenAIModelStep()
manual_context.set_current_step(register_step.step_name, 4)
registered = register_step.execute(classified.output, manual_context)
show_step_result("Evaluation and model record step", registered)
registered.output


### Evaluation and model record step

Registered GenAI model record `websensors_flow_dmoz_health_openrouter_classifier` with macro F1 0.393.

,value
rows,4.000000e+01
classes,4.000000e+00
accuracy,4.000000e-01
precision_macro,3.955357e-01
recall_macro,4.000000e-01
f1_macro,3.930556e-01
precision_weighted,3.955357e-01
recall_weighted,4.000000e-01
f1_weighted,3.930556e-01
gen_ai_request_count,4.000000e+01


,value
registered_model_name,websensors_flow_dmoz_health_openrouter_classifier
external_model_name,openai/gpt-4o-mini
provider_name,openrouter
model_family,sklearn_compatible_genai_classifier
task,genai_text_classification


RegisteredGenAIModel(model_name='websensors_flow_dmoz_health_openrouter_classifier', provider_name='openrouter', model_artifact_path=PosixPath('/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/artifacts/03_classify_with_llm/genai_classifier_sklearn_wrapper.joblib'), model_card_path=PosixPath('/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/artifacts/04_evaluate_and_register_model/final_genai_model_card.json'), evaluation_dataset_path=PosixPath('/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/artifacts/04_evaluate_and_register_model/final_evaluation_dataset.csv'), metrics_path=PosixPath('/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/artifacts/04_evaluate_and_register_model/final_metrics.json'), confusion_matrix_path=PosixPath('/home/marcacini/Documents/PROJETOS/websensors-f

## 12. Execute the full WebSensors Flow pipeline

The complete pipeline is now executed through WebSensors Flow.

The notebook only builds the pipeline from `FlowSettings`, adds the steps, and calls `pipeline.run()`.

All observability data comes from the `StepResult` objects returned by the steps.


In [36]:
os.environ.setdefault("MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING", "true")

pipeline = build_pipeline_from_settings(settings)
pipeline.add(IngestDataStep())
pipeline.add(PreprocessTextStep())
pipeline.add(ClassifyWithLLMStep())
pipeline.add(EvaluateAndRegisterGenAIModelStep())

run_result = pipeline.run()

print(f"Pipeline status: {run_result.report.status}")
print(f"Run id: {run_result.report.run_id}")
print(f"Observability tracking URI: {settings.observability.mlflow.tracking_uri}")
print(f"Experiment: {settings.observability.mlflow.experiment_name}")


───────────────────────────────────────────────────────────────────────────────── WebSensors Flow ──────────────────────────────────────────────────────────────────────────────────

Resolved configuration
+-----------------------------------------------------------------------------------------------------------------------------------+
| Item              | Value                                                                                                         |
|-------------------+---------------------------------------------------------------------------------------------------------------|
| Config            | -                                                                                                             |
| Run ID            | 60372926d4234089b4bac11f9e4c305b                                                                              |
| Project           | websensors-flow-dmoz-health-genai-notebook 1.1.0                                                              |
| Environment       | notebook                                                                                                      |
| Reports           | /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/reports |
| Graylog           | disabled tcp://-:-                                                                                            |
| MLflow            | enabled http://127.0.0.1:5000                                                                                 |
| MLflow experiment | websensors-flow-dmoz-health-genai-notebook                                                                    |
| API               | disabled 127.0.0.1:8000/runs                                                                                  |
| Observers         | console, report, mlflow                                                                                       |
|-------------------+---------------------------------------------------------------------------------------------------------------|
| Steps             | ingest_data -> preprocess_text -> classify_with_llm -> evaluate_and_register_model                            |
+-----------------------------------------------------------------------------------------------------------------------------------+

──────────────────────────────────────────────────────────────────────────────────── Preflight ─────────────────────────────────────────────────────────────────────────────────────

+-------------------------------------- RUNNING preflight ---------------------------------------+
| Status        RUNNING                                                                          |
| Scope         preflight                                                                        |
| Name          console                                                                          |
| Message       Checking observer configuration.                                                 |
| phase         validate_ready                                                                   |
| target        terminal                                                                         |
| duration      -                                                                                |
+------------------------------------------------------------------------------------------------+

+----------------------------------------- OK preflight -----------------------------------------+
| Status        OK                                                                               |
| Scope         preflight                                                                        |
| Name          console                                                                          |
| Message       Observer configuration is ready.                                                 |
| phase         validate_ready                                                                   |
| target        terminal                                                                         |
| duration      0.002s                                                                           |
+------------------------------------------------------------------------------------------------+

...

Records emitted by evaluate_and_register_model
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Type          | Params                                                                          | Metrics                                                                        |
|---------------+---------------------------------------------------------------------------------+--------------------------------------------------------------------------------|
| final_model   | model_name=websensors_flow_dmoz_health_openrouter_classifier;                   | accuracy=0.4; precision_macro=0.3955357142857143; recall_macro=0.4;            |
|               | model_family=sklearn_compatible_genai_classifier; model_interface=fit_predict;  | f1_macro=0.39305555555555555; precision_weighted=0.39553571428571427;          |
|               | external_model_name=openai/gpt-4o-mini; +6 more                                 | recall_weighted=0.4; +14 more                                                  |
| class_metrics | class_name=Addictions;                                                          | precision=0.375; recall=0.3; f1_score=0.3333333333333333; support=10.0         |
|               | model_name=websensors_flow_dmoz_health_openrouter_classifier                    |                                                                                |
| class_metrics | class_name=Alternative;                                                         | precision=0.25; recall=0.2; f1_score=0.2222222222222222; support=10.0          |
|               | model_name=websensors_flow_dmoz_health_openrouter_classifier                    |                                                                                |
| class_metrics | class_name=Animal; model_name=websensors_flow_dmoz_health_openrouter_classifier | precision=0.6; recall=0.6; f1_score=0.6; support=10.0                          |
| class_metrics | class_name=Conditions;                                                          | precision=0.35714285714285715; recall=0.5; f1_score=0.4166666666666667;        |
|               | model_name=websensors_flow_dmoz_health_openrouter_classifier                    | support=10.0                                                                   |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+

Progress  4/4 (100%)

+-------------------------------------------- OK pipeline ---------------------------------------------+
| Status              OK                                                                               |
| Scope               pipeline                                                                         |
| Name                websensors-flow-dmoz-health-genai-notebook                                       |
| Message             Pipeline completed successfully.                                                 |
| duration            27.120s                                                                          |
| pipeline_success    1                                                                                |
| pipeline_failed     0                                                                                |
| steps_total         4                                                                                |
| steps_success       4                                                                                |
| steps_failed        0                                                                                |
| metrics_count       44                                                                               |
| params_count        16                                                                               |
| artifacts_count     17                                                                               |
| metric_records_count7                                                                                |
| warnings_count      0                                                                                |
+------------------------------------------------------------------------------------------------------+

Reports: /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_genai_notebook_workspace/reports

Pipeline status: success
Run id: 60372926d4234089b4bac11f9e4c305b
Observability tracking URI: http://127.0.0.1:5000
Experiment: websensors-flow-dmoz-health-genai-notebook


## 13. Inspect the pipeline report

This local report is useful for checking what each step emitted before looking at the observability UI.

The expected result is:

- one dataset record from `ingest_data`;
- one model candidate record from `classify_with_llm`;
- one final model record from `evaluate_and_register_model`;
- class-level metric records;
- no row-level metric records.


In [37]:
step_rows = []
for step in run_result.report.steps:
    step_rows.append(
        {
            "step": step.step_name,
            "status": step.status,
            "duration_seconds": step.duration_seconds,
            "metrics": step.metrics,
            "params": step.params,
            "artifacts": step.artifacts,
        }
    )

report_df = pd.DataFrame(step_rows)
report_df


,step,status,duration_seconds,metrics,params,artifacts
0,ingest_data,success,0.302373,"{'rows': 40, 'classes': 4, 'min_class_count': ...",{'dataset_url': 'https://raw.githubusercontent...,{'raw_dataset': '/home/marcacini/Documents/PRO...
1,preprocess_text,success,0.111244,"{'rows_before_filter': 40, 'rows_after_filter'...","{'min_text_length': 20, 'text_column': 'text',...",{'prepared_dataset': '/home/marcacini/Document...
2,classify_with_llm,success,0.109330,"{'prediction_rows': 40, 'gen_ai_request_count'...",{'registered_model_name': 'websensors_flow_dmo...,{'model_artifact': '/home/marcacini/Documents/...
3,evaluate_and_register_model,success,0.116743,"{'rows': 40, 'classes': 4, 'accuracy': 0.4, 'p...",{'registered_model_name': 'websensors_flow_dmo...,{'model_bundle': '/home/marcacini/Documents/PR...


## 14. Test the serialized GenAI classifier artifact

The saved model artifact contains the scikit-learn-compatible wrapper. Loading the artifact and calling `predict` should work like a normal classifier.

When the model was saved in mock mode, it can predict without an API key. When it was saved in real OpenRouter mode, it still reads the key from the configured environment variable name.


In [38]:
model_bundle = joblib.load(registered.output.model_artifact_path)
loaded_classifier = model_bundle["model"]

sample_texts = [
    "A hospital page about diagnosis, treatment, and patient care.",
    "A guide with workout routines, running plans, and exercise tips.",
]

loaded_classifier.predict(sample_texts)


array(['Conditions', 'Addictions'], dtype='<U10')

## 15. Open the observability UI

Start the UI using the same tracking URI configured in `FlowSettings`.

In the experiment, check the run generated by the full pipeline. The expected observability objects are the dataset, the serialized GenAI classifier model artifact, aggregated GenAI metrics, prompt artifacts, prediction artifacts, and evaluation artifacts.


In [39]:
print("Command to open the observability UI:")
print(f"mlflow ui --backend-store-uri {settings.observability.mlflow.tracking_uri}")
print("\nExperiment name:")
print(settings.observability.mlflow.experiment_name)


Command to open the observability UI:
mlflow ui --backend-store-uri http://127.0.0.1:5000

Experiment name:
websensors-flow-dmoz-health-genai-notebook
